# Module 9B — Entity Linking: News Headlines Demo

**Session:** Day 2 | **Duration:** 90 minutes

A complete entity linking pipeline connecting unstructured news headlines to a Neo4j Knowledge Graph. Covers four Cypher beats: parameterized candidate retrieval, NER type filtering, `[:SUBCLASS_OF*0..]` hierarchical traversal, and NIL handling with reason plumbing.

**Prerequisites:**
```
pip install spacy neo4j
python -m spacy download en_core_web_sm
```
Neo4j must be running at `bolt://localhost:7687`.

## Section 1 — Setup & Orientation

Import dependencies, connect to Neo4j, confirm the geo-graph schema is present (labels, relationship types), and print a preview of the four Cypher beats we will implement.

In [1]:
# ── Imports ────────────────────────────────────────────────────────────────
import spacy                            # NLP pipeline for named-entity recognition
from neo4j import GraphDatabase         # Official Neo4j Python driver
from dataclasses import dataclass, field  # Typed result container
from typing import Optional, Set, Dict, List  # Type hints

# ── 1a. Load the spaCy English model ───────────────────────────────────────
nlp = spacy.load("en_core_web_sm")      # Small English pipeline; includes NER
print("spaCy model loaded:", nlp.meta["name"])

# ── 1b. Connect to Neo4j ───────────────────────────────────────────────────
NEO4J_URI  = "bolt://localhost:7687"    # Default Bolt port for local Neo4j
NEO4J_USER = "neo4j"                   # Default superuser account
NEO4J_PASS = "password"                # Replace with your actual password

driver = GraphDatabase.driver(          # Creates a reusable connection pool
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASS)
)
driver.verify_connectivity()            # Raises immediately if server is unreachable
print("Neo4j connection established.")

# ── 1c. Inspect the geo-graph schema ──────────────────────────────────────
with driver.session() as session:
    labels = session.run("CALL db.labels() YIELD label RETURN label").value()
    print("KG labels:", labels)         # Expect: Entity, City, Country, Continent …

    rel_types = session.run(
        "CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType"
    ).value()
    print("KG relationships:", rel_types)  # Expect: SUBCLASS_OF

# ── 1d. Preview the four Cypher beats (orientation only — not executed yet)
BEAT_PREVIEWS = [
    "MATCH (n:Entity) WHERE toLower(n.name) = toLower($surface) RETURN n.id, n.name, labels(n)",
    "# NER label → KG type dict  →  filter candidates by allowed_types",
    "MATCH (c:Entity {id: $cand_id})-[:SUBCLASS_OF*0..]->(ancestor:Entity) RETURN collect(labels(ancestor))",
    "# Return LinkResult(..., predicted_node_id=None, reason='nil-no-candidates')",
]
for i, beat in enumerate(BEAT_PREVIEWS, 1):
    print(f"Beat {i}: {beat}")

/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


spaCy model loaded: core_web_sm
Neo4j connection established.
KG labels: ['Entity', 'City', 'Mountain', 'Organization', 'Company', 'Country', 'Continent']
KG relationships: ['SUBCLASS_OF']
Beat 1: MATCH (n:Entity) WHERE toLower(n.name) = toLower($surface) RETURN n.id, n.name, labels(n)
Beat 2: # NER label → KG type dict  →  filter candidates by allowed_types
Beat 3: MATCH (c:Entity {id: $cand_id})-[:SUBCLASS_OF*0..]->(ancestor:Entity) RETURN collect(labels(ancestor))
Beat 4: # Return LinkResult(..., predicted_node_id=None, reason='nil-no-candidates')


## Section 2 — spaCy NER on a Headline

Run the spaCy NER pipeline on two ambiguous headlines. Print each entity's surface text, NER label, and character offsets. The output illustrates what disambiguation must resolve: `Paris` (city vs. person) and `Apple` (company vs. fruit).

In [2]:
# ── Two demo headlines with deliberate ambiguity ───────────────────────────
headlines = [
    "Paris hosts the climate summit",    # Ambiguous: city vs. person
    "Apple unveils new device",          # Ambiguous: company vs. fruit
]

# ── Run spaCy NER on each headline ────────────────────────────────────────
for headline in headlines:
    doc = nlp(headline)                 # Full spaCy pipeline: tokenize → POS → NER
    print(f"\nHeadline: {headline!r}")
    print(f"  Tokens : {[t.text for t in doc]}")  # All tokens in the sentence

    if doc.ents:                        # .ents holds all recognized entity spans
        for ent in doc.ents:
            print(
                f"  Entity : {ent.text!r:20s} "
                f"label={ent.label_:10s} "        # NER category (GPE, ORG, PERSON …)
                f"char=[{ent.start_char}:{ent.end_char}]"  # Character offsets
            )
    else:
        print("  (no entities detected)")

# ── Note the disambiguation problem for each span ─────────────────────────
print("\n--- Disambiguation note ---")
print("'Paris' → GPE: could be the city (in our KG) or a celebrity (not in KG).")
print("'Apple' → ORG: the NER label rules out the fruit; company still needs KG lookup.")
print("The linker must pick the single best KG node, or return NIL.")


Headline: 'Paris hosts the climate summit'
  Tokens : ['Paris', 'hosts', 'the', 'climate', 'summit']
  Entity : 'Paris'              label=GPE        char=[0:5]

Headline: 'Apple unveils new device'
  Tokens : ['Apple', 'unveils', 'new', 'device']
  Entity : 'Apple'              label=ORG        char=[0:5]

--- Disambiguation note ---
'Paris' → GPE: could be the city (in our KG) or a celebrity (not in KG).
'Apple' → ORG: the NER label rules out the fruit; company still needs KG lookup.
The linker must pick the single best KG node, or return NIL.


## Section 3 — Candidates via Parameterized Cypher

Query Neo4j for every node whose `.name` matches the surface form (case-insensitive). The query uses a `$surface` parameter placeholder — not an f-string. The autograder's AST check rejects f-string Cypher; the anti-pattern is shown in comments for contrast.

In [3]:
# ── Parameterized candidate-retrieval query ────────────────────────────────
CANDIDATE_QUERY = """
MATCH (n:Entity)
WHERE toLower(n.name) = toLower($surface)
RETURN n.id AS id, n.name AS name, labels(n) AS kg_labels
"""
# $surface is a Cypher parameter — Neo4j substitutes it safely after compilation

def get_candidates(session, surface: str) -> List[Dict]:
    """Return all KG nodes whose name matches the surface form."""
    result = session.run(
        CANDIDATE_QUERY,
        surface=surface          # Named keyword arg → maps to $surface in Cypher
    )
    return [dict(record) for record in result]  # Collapse cursor to plain dicts

# ── Test with surface form "Paris" ─────────────────────────────────────────
with driver.session() as session:
    candidates = get_candidates(session, "Paris")

print(f"Candidates for 'Paris' ({len(candidates)} found):")
for c in candidates:
    print(f"  {c}")

# ── Anti-pattern (do NOT do this) ─────────────────────────────────────────
# WRONG — f-string injection risk:
# bad_query = f"MATCH (n:Entity) WHERE n.name = '{surface}' RETURN n"
# session.run(bad_query)  ← input "x' OR 1=1 //" would return all nodes
#
# RIGHT — always pass values as keyword arguments to session.run():
# session.run(CANDIDATE_QUERY, surface=surface)

print("\nRule: session.run(query, param=value) — never f-strings in Cypher.")

Candidates for 'Paris' (1 found):
  {'id': 'Q90', 'name': 'Paris', 'kg_labels': ['Entity', 'City']}

Rule: session.run(query, param=value) — never f-strings in Cypher.


## Section 4 — NER → KG Type-Filter Disambiguation

Define the `LinkResult` dataclass (the single return type for the entire linker) and `NER_LABEL_TO_KG_TYPE` (the bridge between spaCy labels and KG node-label sets). Use set intersection to prune candidates. If exactly one survives, return it with `reason="resolved-by-type"`.

In [4]:
# ── LinkResult — the single return type for the entire linker ──────────────
@dataclass
class LinkResult:
    surface: str                             # Original surface form from NER
    ner_label: str                           # spaCy NER label (GPE, ORG …)
    predicted_node_id: Optional[str]         # Winning KG node id, or None for NIL
    reason: str                              # Controlled-vocabulary diagnostic token
    candidates: List[Dict] = field(default_factory=list)  # All candidates pre-filter

# ── NER label → allowed KG node-label set (the NER/KG bridge) ─────────────
NER_LABEL_TO_KG_TYPE: Dict[str, Set[str]] = {
    "GPE":    {"City", "Country", "Region"},        # Geo-political entities
    "ORG":    {"Organization", "Company"},          # Corporate / institutional
    "PERSON": {"Person"},                           # Named human individuals
    "LOC":    {"Continent", "Ocean", "Mountain"},   # Non-GPE physical locations
    "PRODUCT": {"Product"},                         # Manufactured items
}

def type_filter(candidates: List[Dict], ner_label: str) -> List[Dict]:
    """Keep only candidates whose KG labels intersect the allowed set."""
    allowed = NER_LABEL_TO_KG_TYPE.get(ner_label, set())  # Empty set if label unknown
    return [
        c for c in candidates
        if set(c["kg_labels"]) & allowed   # Set intersection: at least one label matches
    ]

def disambiguate_type(session, surface: str, ner_label: str) -> LinkResult:
    """Stage-1 disambiguation: candidate retrieval + type filter."""
    candidates = get_candidates(session, surface)  # Beat 1: fetch all candidates
    filtered   = type_filter(candidates, ner_label)  # Beat 2: prune by NER type

    if len(filtered) == 1:                          # Unique survivor → resolved
        return LinkResult(
            surface=surface,
            ner_label=ner_label,
            predicted_node_id=filtered[0]["id"],
            reason="resolved-by-type",              # Diagnostic token
            candidates=candidates,
        )
    # Zero or multiple survivors → escalate to hierarchy (Sections 5–6)
    return LinkResult(
        surface=surface,
        ner_label=ner_label,
        predicted_node_id=None,
        reason="needs-hierarchy" if filtered else "nil-no-candidates",
        candidates=candidates,
    )

# ── Demo ──────────────────────────────────────────────────────────────────
with driver.session() as session:
    print("Paris (GPE):", disambiguate_type(session, "Paris", "GPE"))
    print("Apple (ORG):", disambiguate_type(session, "Apple", "ORG"))

Paris (GPE): LinkResult(surface='Paris', ner_label='GPE', predicted_node_id='Q90', reason='resolved-by-type', candidates=[{'id': 'Q90', 'name': 'Paris', 'kg_labels': ['Entity', 'City']}])
Apple (ORG): LinkResult(surface='Apple', ner_label='ORG', predicted_node_id='Q312', reason='resolved-by-type', candidates=[{'id': 'Q312', 'name': 'Apple', 'kg_labels': ['Entity', 'Organization', 'Company']}])


## Section 5 — Hierarchical Traversal with `[:SUBCLASS_OF*0..]`

When the direct KG label of a candidate does not match the allowed type set, walk the taxonomy upward via `[:SUBCLASS_OF*0..]` (zero-or-more hops). If any ancestor is type-compatible, resolve with `reason="resolved-by-hierarchy"`. This is the load-bearing beat for cuisine terms like "Sichuan" in the recipe assignment.

In [5]:
# ── Hierarchical traversal query ───────────────────────────────────────────
HIERARCHY_QUERY = """
MATCH (c:Entity {id: $cand_id})-[:SUBCLASS_OF*0..]->(ancestor:Entity)
RETURN collect(labels(ancestor)) AS ancestor_label_sets
"""
# *0.. = zero or more hops (includes the node itself at 0 hops)
# collect() aggregates all ancestor rows into one list on the driver side

def get_ancestor_labels(session, cand_id: str) -> Set[str]:
    """Return the flat union of all labels across every ancestor of cand_id."""
    result = session.run(HIERARCHY_QUERY, cand_id=cand_id)  # Parameterized
    record = result.single()                                 # One aggregated row
    if record is None:
        return set()                                         # Node not found
    ancestor_labels: Set[str] = set()
    for label_list in record["ancestor_label_sets"]:         # List-of-lists from collect()
        ancestor_labels.update(label_list)                   # Flatten into one set
    return ancestor_labels

def disambiguate_hierarchy(session, surface: str, ner_label: str) -> LinkResult:
    """Stage-2 disambiguation: fall back to hierarchy if type filter is inconclusive."""
    candidates = get_candidates(session, surface)
    filtered   = type_filter(candidates, ner_label)

    if len(filtered) == 1:                                   # Stage 1 resolved it
        return LinkResult(
            surface=surface, ner_label=ner_label,
            predicted_node_id=filtered[0]["id"],
            reason="resolved-by-type", candidates=candidates
        )

    # Walk the taxonomy for each remaining candidate
    allowed = NER_LABEL_TO_KG_TYPE.get(ner_label, set())
    hierarchy_matches = [
        c for c in candidates
        if get_ancestor_labels(session, c["id"]) & allowed  # Any ancestor type-compatible?
    ]

    if len(hierarchy_matches) == 1:                          # Unique hierarchy resolution
        return LinkResult(
            surface=surface, ner_label=ner_label,
            predicted_node_id=hierarchy_matches[0]["id"],
            reason="resolved-by-hierarchy",
            candidates=candidates
        )

    return LinkResult(
        surface=surface, ner_label=ner_label,
        predicted_node_id=None,
        reason="nil-ambiguous" if hierarchy_matches else "nil-no-candidates",
        candidates=candidates
    )

# ── Demo: LOC tag whose direct label is :City but ancestor is :Continent ──
with driver.session() as session:
    result = disambiguate_hierarchy(session, "Paris", "LOC")
    print("Paris (LOC via hierarchy):", result)
    # Paris → France → Europe → Continent → resolved-by-hierarchy

Paris (LOC via hierarchy): LinkResult(surface='Paris', ner_label='LOC', predicted_node_id='Q90', reason='resolved-by-hierarchy', candidates=[{'id': 'Q90', 'name': 'Paris', 'kg_labels': ['Entity', 'City']}])


## Section 6 — NIL Handling + Reason Plumbing

Combine all stages into a single `link_entity()` function. Two distinct NIL cases are distinguished by their reason token: `"nil-no-candidates"` (surface absent from KG) and `"nil-ambiguous"` (multiple survivors, no resolving signal). The exact token strings are part of a public contract — the integration repo's diagnostics grep for them.

In [6]:
# ── Full 4-stage linker ────────────────────────────────────────────────────

def link_entity(session, surface: str, ner_label: str) -> LinkResult:
    """
    Stage 1 — candidate retrieval (parameterized Cypher)
    Stage 2 — NER type filter
    Stage 3 — [:SUBCLASS_OF*0..] hierarchy traversal
    Stage 4 — NIL with reason token
    """
    # Stage 1: retrieve all name-matching candidates
    candidates = get_candidates(session, surface)

    if not candidates:                              # Surface absent from KG entirely
        return LinkResult(
            surface=surface, ner_label=ner_label,
            predicted_node_id=None,
            reason="nil-no-candidates",             # Exact token — do not alter
            candidates=[]
        )

    # Stage 2: prune by NER type
    filtered = type_filter(candidates, ner_label)

    if len(filtered) == 1:                          # Unique type match → done
        return LinkResult(
            surface=surface, ner_label=ner_label,
            predicted_node_id=filtered[0]["id"],
            reason="resolved-by-type",
            candidates=candidates
        )

    # Stage 3: hierarchy on surviving candidates (widen to all if filter cleared none)
    pool    = filtered if filtered else candidates
    allowed = NER_LABEL_TO_KG_TYPE.get(ner_label, set())
    hierarchy_matches = [
        c for c in pool
        if get_ancestor_labels(session, c["id"]) & allowed
    ]

    if len(hierarchy_matches) == 1:                 # Unique hierarchy match → done
        return LinkResult(
            surface=surface, ner_label=ner_label,
            predicted_node_id=hierarchy_matches[0]["id"],
            reason="resolved-by-hierarchy",
            candidates=candidates
        )

    # Stage 4: give up — distinguish the two NIL cases by reason token
    return LinkResult(
        surface=surface, ner_label=ner_label,
        predicted_node_id=None,
        reason="nil-ambiguous" if hierarchy_matches else "nil-no-candidates",
        candidates=candidates
    )

# ── Demo both NIL cases ────────────────────────────────────────────────────
nil_demos = [
    ("Zephyria",    "GPE", "Expected: nil-no-candidates (fictional city)"),
    ("Springfield", "GPE", "Expected: nil-ambiguous (many cities, no resolver)"),
]

with driver.session() as session:
    for surface, ner_label, note in nil_demos:
        result = link_entity(session, surface, ner_label)
        print(f"\n{note}")
        print(f"  reason={result.reason!r}  predicted={result.predicted_node_id!r}")


Expected: nil-no-candidates (fictional city)
  reason='nil-no-candidates'  predicted=None

Expected: nil-ambiguous (many cities, no resolver)
  reason='nil-ambiguous'  predicted=None


## Section 7 — P/R/F1 over a 10-Headline Gold Set

Score the linker against a gold standard of `(surface, ner_label, gold_node_id)` triples, where `None` means the correct answer is NIL. Four cases: TP, FP, FN, TN-NIL. Note the bug: a correct NIL prediction must be counted as TN-NIL, **not** FN. The autograder's four-case fixture pins this behavior.

In [7]:
# ── Gold standard: 10 (surface, ner_label, gold_node_id) triples ──────────
# None in gold_node_id = correct answer is NIL
GOLD_SET = [
    ("Paris",         "GPE",  "Q90"),     # TP
    ("London",        "GPE",  "Q84"),     # TP
    ("Apple",         "ORG",  "Q312"),    # TP
    ("Zephyria",      "GPE",  None),      # TN-NIL: fictional city
    ("Springfield",   "GPE",  None),      # TN-NIL: ambiguous, gold=NIL
    ("Berlin",        "GPE",  "Q64"),     # TP
    ("Tokyo",         "GPE",  "Q1490"),   # TP
    ("FakeCorp",      "ORG",  None),      # TN-NIL: not in KG
    ("Amazon",        "ORG",  "Q3884"),   # FP risk: multiple ORG candidates
    ("Mount Olympus", "LOC",  "Q182212"), # Tests hierarchy (LOC → Continent path)
]

def score(gold_set, session) -> Dict[str, float]:
    """Compute P/R/F1. Correct NIL predictions are TN-NIL, not FN."""
    tp = fp = fn = tn_nil = 0                   # Four outcome counters

    for surface, ner_label, gold_id in gold_set:
        pred_id = link_entity(session, surface, ner_label).predicted_node_id

        if gold_id is None and pred_id is None:
            tn_nil += 1                          # Correct abstention — not penalized
        elif gold_id is not None and pred_id == gold_id:
            tp += 1                              # Correct non-NIL link
        elif gold_id is not None and pred_id != gold_id:
            fn += 1                              # Missed the correct entity
        else:                                    # pred not None, gold is None
            fp += 1                              # Linked when should have been NIL

    # Standard micro P/R/F1 — tn_nil does not appear in any formula
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        "TP": tp, "FP": fp, "FN": fn, "TN-NIL": tn_nil,
        "Precision": round(precision, 4),
        "Recall":    round(recall, 4),
        "F1":        round(f1, 4),
    }

with driver.session() as session:
    metrics = score(GOLD_SET, session)

print("--- Evaluation results ---")
for k, v in metrics.items():
    print(f"  {k}: {v}")

--- Evaluation results ---
  TP: 7
  FP: 0
  FN: 0
  TN-NIL: 3
  Precision: 1.0
  Recall: 1.0
  F1: 1.0


## Section 8 — Pivot to Assignment

Map every demo concept to its recipe-corpus equivalent. Print the four TODO function signatures, deliverable requirements, evaluation thresholds, and the exact reason token vocabulary required by the autograder and integration diagnostics.

In [8]:
# ── Assignment concept mapping ─────────────────────────────────────────────
print("=" * 60)
print("ASSIGNMENT PIVOT SUMMARY")
print("=" * 60)

mapping = {
    "Dataset":           "200-doc recipe corpus (not 10 headlines)",
    "KG node types":     "Recipe, Cuisine, Ingredient, Author, Technique",
    "Key NER labels":    "CUISINE, INGREDIENT, PERSON (author)",
    "Hierarchy example": "Sichuan → Chinese → Asian → World",
    "Cypher beat 3":     "[:SUBCLASS_OF*0..] — identical Cypher, different graph",
}
for concept, equivalent in mapping.items():
    print(f"  {concept:20s}: {equivalent}")

# ── Four TODO functions in linker.py ──────────────────────────────────────
print("\nTODO functions to implement in linker.py:")
todos = [
    "get_candidates(session, surface)          → parameterized Cypher",
    "type_filter(candidates, ner_label)        → extend NER_LABEL_TO_KG_TYPE for recipes",
    "get_ancestor_labels(session, cand_id)     → [:SUBCLASS_OF*0..] traversal",
    "disambiguate(session, surface, ner_label) → orchestrate all stages → LinkResult",
]
for i, todo in enumerate(todos, 1):
    print(f"  {i}. {todo}")

# ── Evaluation thresholds ──────────────────────────────────────────────────
print("\nTest-split thresholds (partial-cascade implementation passes all three):")
print("  Precision ≥ 0.80")
print("  Recall    ≥ 0.65")
print("  F1        ≥ 0.72")

# ── Reason token vocabulary — exact strings, do not alter ─────────────────
print("\nReason token vocabulary (integration diagnostics grep for these exactly):")
for token in ["resolved-by-type", "resolved-by-hierarchy", "nil-no-candidates", "nil-ambiguous"]:
    print(f"  {token!r}")

ASSIGNMENT PIVOT SUMMARY
  Dataset             : 200-doc recipe corpus (not 10 headlines)
  KG node types       : Recipe, Cuisine, Ingredient, Author, Technique
  Key NER labels      : CUISINE, INGREDIENT, PERSON (author)
  Hierarchy example   : Sichuan → Chinese → Asian → World
  Cypher beat 3       : [:SUBCLASS_OF*0..] — identical Cypher, different graph

TODO functions to implement in linker.py:
  1. get_candidates(session, surface)          → parameterized Cypher
  2. type_filter(candidates, ner_label)        → extend NER_LABEL_TO_KG_TYPE for recipes
  3. get_ancestor_labels(session, cand_id)     → [:SUBCLASS_OF*0..] traversal
  4. disambiguate(session, surface, ner_label) → orchestrate all stages → LinkResult

Test-split thresholds (partial-cascade implementation passes all three):
  Precision ≥ 0.80
  Recall    ≥ 0.65
  F1        ≥ 0.72

Reason token vocabulary (integration diagnostics grep for these exactly):
  'resolved-by-type'
  'resolved-by-hierarchy'
  'nil-no-candidates'
 

## Section 9 — Shutdown

Close the Neo4j driver to release the connection pool, then stop and remove the Docker containers defined in `docker-compose.yml`. Run this cell when finished to free system resources.

In [ ]:
import subprocess                           # Run shell commands from Python
from pathlib import Path                    # Cross-platform path construction

# ── 9a. Close the Neo4j driver ─────────────────────────────────────────────
driver.close()                              # Returns all pooled connections; no-op if already closed
print("Neo4j driver closed.")

# ── 9b. Stop and remove the Docker containers ──────────────────────────────
LAB_DIR = Path.home() / "Documents/brandon/Istidama Bootcamp/Modules/Module 9B Applied Lab"

result = subprocess.run(
    ["docker", "compose", "down"],          # Stops containers and removes them
    cwd=LAB_DIR,                            # Must run from the directory containing docker-compose.yml
    capture_output=True,                    # Collect stdout and stderr
    text=True,                              # Decode bytes as UTF-8 strings
)

print(result.stdout or "(no stdout)")
if result.returncode != 0:                  # Non-zero exit means something failed
    print("stderr:", result.stderr)
else:
    print("All containers stopped and removed.")